# Predição de Doença Cardíaca — Árvore de Decisão

Dataset: [Heart Disease Dataset (johnsmith88) — Kaggle](https://www.kaggle.com/datasets/johnsmith88/heart-disease-dataset)

Este notebook executa: carga dos dados, análise exploratória rápida, pré-processamento, treinamento de um modelo de Árvore de Decisão, avaliação com métricas de classificação e visualização da importância das variáveis.

## 1. Upload do dataset

Rode a célula abaixo e selecione o arquivo `heart.csv` quando solicitado.

In [ ]:
from google.colab import files
uploaded = files.upload()  # selecione o arquivo heart.csv

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('heart.csv')
df.head()

## 2. Análise exploratória (EDA) rápida

In [ ]:
print("Dimensões:", df.shape)
print("\nValores nulos por coluna:")
print(df.isnull().sum())
print("\nLinhas duplicadas:", df.duplicated().sum())
print("\nDistribuição da variável target:")
print(df['target'].value_counts())

df.describe()

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(data=df, x='target')
plt.title('Distribuição de pacientes com e sem doença cardíaca')
plt.xlabel('0 = Sem doença | 1 = Com doença')
plt.show()

plt.figure(figsize=(10,8))
sns.heatmap(df.corr(), annot=False, cmap='coolwarm')
plt.title('Matriz de correlação entre variáveis')
plt.show()

## 3. Pré-processamento e divisão treino/teste

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['target'])
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Treino:", X_train.shape, "| Teste:", X_test.shape)

## 4. Treinamento do modelo — Árvore de Decisão (Decision Tree Classifier)

A Árvore de Decisão aprende uma sequência de perguntas do tipo "se/então" sobre as variáveis (ex.: "o tipo de dor no peito é igual a X? Se sim, vá para a direita; senão, para a esquerda"), dividindo os pacientes em grupos cada vez mais puros (com predominância de uma única classe), até chegar a uma folha com a decisão final. Limitamos a profundidade (`max_depth`) para evitar que a árvore memorize demais os dados de treino (overfitting).

In [ ]:
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(max_depth=5, random_state=42)
model.fit(X_train, y_train)

## 5. Execução dos testes e métricas

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("Acurácia: ", round(accuracy_score(y_test, y_pred), 4))
print("Precisão: ", round(precision_score(y_test, y_pred), 4))
print("Recall:   ", round(recall_score(y_test, y_pred), 4))
print("F1-score: ", round(f1_score(y_test, y_pred), 4))
print("ROC-AUC:  ", round(roc_auc_score(y_test, y_proba), 4))
print("\nRelatório de classificação:\n")
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Sem doença','Com doença'],
            yticklabels=['Sem doença','Com doença'])
plt.xlabel('Predito')
plt.ylabel('Real')
plt.title('Matriz de Confusão')
plt.show()

## 6. Visualização da árvore e importância das variáveis

In [ ]:
from sklearn.tree import plot_tree

plt.figure(figsize=(20,10))
plot_tree(model, feature_names=X.columns, class_names=['Sem doença','Com doença'],
          filled=True, max_depth=3, fontsize=9)
plt.title('Árvore de Decisão (3 primeiros níveis)')
plt.show()

In [ ]:
importancias = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(8,5))
sns.barplot(x=importancias.values, y=importancias.index, color='steelblue')
plt.title('Importância das variáveis (Árvore de Decisão)')
plt.xlabel('Importância')
plt.show()

importancias

## 7. Resultados da primeira execução (registro)

In [ ]:
resultados = {
    'Acuracia': accuracy_score(y_test, y_pred),
    'Precisao': precision_score(y_test, y_pred),
    'Recall': recall_score(y_test, y_pred),
    'F1_score': f1_score(y_test, y_pred),
    'ROC_AUC': roc_auc_score(y_test, y_proba)
}
pd.DataFrame([resultados])